In [1]:
# ============================================================
# Notebook    : 03a_lightgbm_static.ipynb
# Description : Case A extension — Model ① LightGBM-Static
#               Static variables ONLY: Expo, Usage, VehType, VehPower
#               NO YearGap, NO sequence structure.
#               - Row-level (timestep) tabular input
#               - IDpol-level train/val/test split reused from
#                 notebook 02 (data/sequences/split_idpols.json)
#               - Evaluated at timestep level (flattened rows,
#                 same convention as 03/04b Transformer) so all
#                 four models (①②③④) are directly comparable
# ============================================================


# ============================================================
# 0. Install dependencies (run once)
# ============================================================
# pip install lightgbm scikit-learn pandas numpy


# ============================================================
# 1. Common imports
# ============================================================
import json
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, f1_score, precision_recall_curve

SEED = 42
np.random.seed(SEED)


# ============================================================
# 2. Load preprocessed data (from notebook 01) and the
#    IDpol-level split (from notebook 02) — this guarantees
#    Model ① sees the EXACT SAME test population as ②③④
# ============================================================
df = pd.read_csv("data/fremotor_multi_history_features.csv")
print(f"[CHECK 1] df shape: {df.shape}")

with open("data/sequences/split_idpols.json", "r", encoding="utf-8") as f:
    split_ids = json.load(f)

train_idpols = set(split_ids["train"])
val_idpols   = set(split_ids["val"])
test_idpols  = set(split_ids["test"])

print(f"[CHECK 1] IDpol counts — train: {len(train_idpols):,}, "
      f"val: {len(val_idpols):,}, test: {len(test_idpols):,}")

df_train = df[df["IDpol"].isin(train_idpols)].copy()
df_val   = df[df["IDpol"].isin(val_idpols)].copy()
df_test  = df[df["IDpol"].isin(test_idpols)].copy()

print(f"\n[CHECK 2] Row counts (timestep-level) — "
      f"train: {len(df_train):,}, val: {len(df_val):,}, test: {len(df_test):,}")
print(f"[CHECK 2] Positive rates — "
      f"train: {df_train['Label'].mean()*100:.2f}%, "
      f"val: {df_val['Label'].mean()*100:.2f}%, "
      f"test: {df_test['Label'].mean()*100:.2f}%")


# ============================================================
# 3. Feature scope — STATIC ONLY
#    - NO YearGap (this is exactly what separates ① from ②)
#    - Categorical columns passed as pandas 'category' dtype,
#      LightGBM handles native categorical splits without
#      one-hot encoding
# ============================================================
NUMERIC_COLS = ["Expo"]
CAT_COLS     = ["Usage", "VehType", "VehPower"]
FEATURE_COLS = NUMERIC_COLS + CAT_COLS
LABEL_COL    = "Label"

print(f"\n[CHECK 3] Feature scope (STATIC — no YearGap):")
print(f"  Numeric     : {NUMERIC_COLS}")
print(f"  Categorical : {CAT_COLS}")

for d in (df_train, df_val, df_test):
    for col in CAT_COLS:
        d[col] = d[col].astype("category")

X_train, y_train = df_train[FEATURE_COLS], df_train[LABEL_COL]
X_val,   y_val   = df_val[FEATURE_COLS],   df_val[LABEL_COL]
X_test,  y_test  = df_test[FEATURE_COLS],  df_test[LABEL_COL]

# align categorical dtype categories across splits so LightGBM
# doesn't choke on a category seen in test but not in train
for col in CAT_COLS:
    all_cats = pd.api.types.union_categoricals(
        [X_train[col], X_val[col], X_test[col]]
    ).categories
    X_train[col] = X_train[col].cat.set_categories(all_cats)
    X_val[col]   = X_val[col].cat.set_categories(all_cats)
    X_test[col]  = X_test[col].cat.set_categories(all_cats)

print(f"[CHECK 3] X_train shape: {X_train.shape}")


# ============================================================
# 4. Train LightGBM
#    - pos_weight-equivalent via scale_pos_weight, matching the
#      ~6.89 imbalance ratio used in the Transformer (pos_weight)
#      so all four models correct for the same 12.67% base rate
#      in a comparable way
# ============================================================
POS_RATE = y_train.mean()
SCALE_POS_WEIGHT = (1 - POS_RATE) / POS_RATE
print(f"\n[CHECK 4] scale_pos_weight: {SCALE_POS_WEIGHT:.2f}")

train_set = lgb.Dataset(X_train, label=y_train, categorical_feature=CAT_COLS)
val_set   = lgb.Dataset(X_val,   label=y_val,   categorical_feature=CAT_COLS, reference=train_set)

params = {
    "objective": "binary",
    "metric": "auc",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "scale_pos_weight": SCALE_POS_WEIGHT,
    "seed": SEED,
    "verbose": -1,
}

print(f"[CHECK 4] Training LightGBM-Static...")
model = lgb.train(
    params,
    train_set,
    num_boost_round=500,
    valid_sets=[train_set, val_set],
    valid_names=["train", "val"],
    callbacks=[
        lgb.early_stopping(stopping_rounds=30),
        lgb.log_evaluation(period=50),
    ],
)
print(f"\n[CHECK 4] Best iteration: {model.best_iteration}")


# ============================================================
# 5. Evaluate on test set — timestep-level metrics
#    - Every row in df_test is one (IDpol, Year) timestep,
#      already at the same granularity as the flattened
#      Transformer evaluation in notebook 03/04b
# ============================================================
y_pred_prob = model.predict(X_test, num_iteration=model.best_iteration)
y_pred_cls  = (y_pred_prob >= 0.5).astype(int)

auc = roc_auc_score(y_test, y_pred_prob)
f1  = f1_score(y_test, y_pred_cls)

precisions, recalls, thresholds = precision_recall_curve(y_test, y_pred_prob)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
best_idx  = np.argmax(f1_scores[:-1])

print("\n===== Test set evaluation (timestep-level) =====")
print(f"Total timestep predictions: {len(y_test):,}")
print(f"AUC-ROC          : {auc:.4f}")
print(f"F1 @ thr=0.5      : {f1:.4f}")
print(f"Best threshold    : {thresholds[best_idx]:.3f}")
print(f"Best F1           : {f1_scores[best_idx]:.4f}")
print(f"  Precision       : {precisions[best_idx]:.4f}")
print(f"  Recall          : {recalls[best_idx]:.4f}")
print(f"Positive rate in test: {y_test.mean()*100:.2f}%")


# ============================================================
# 6. Feature importance (quick check — full SHAP deferred to
#    governance notebook, kept here only as a sanity view)
# ============================================================
importance = pd.DataFrame({
    "feature": FEATURE_COLS,
    "gain": model.feature_importance(importance_type="gain"),
}).sort_values("gain", ascending=False)

print("\n===== Feature importance (gain) =====")
print(importance.to_string(index=False))


# ============================================================
# 7. Save model and predictions
# ============================================================
model.save_model("data/sequences/lightgbm_static_model.txt")
print("\nSaved: data/sequences/lightgbm_static_model.txt")

np.savez(
    "data/sequences/lightgbm_static_test_predictions.npz",
    probs=y_pred_prob,
    labels=y_test.values,
)
print("Saved: data/sequences/lightgbm_static_test_predictions.npz")


# ============================================================
# 8. Summary check
# ============================================================
print("\n===== LightGBM-Static (Model ①) Summary =====")
print(f"Feature scope       : {FEATURE_COLS} (STATIC — no YearGap)")
print(f"Train / Val / Test rows: {len(X_train):,} / {len(X_val):,} / {len(X_test):,}")
print(f"Best iteration      : {model.best_iteration}")
print(f"Test AUC-ROC        : {auc:.4f}")
print(f"Test F1 (best thr)  : {f1_scores[best_idx]:.4f}")
print("===============================================")

[CHECK 1] df shape: (364997, 11)
[CHECK 1] IDpol counts — train: 49,974, val: 10,709, test: 10,709

[CHECK 2] Row counts (timestep-level) — train: 255,241, val: 55,001, test: 54,755
[CHECK 2] Positive rates — train: 12.66%, val: 12.63%, test: 12.79%

[CHECK 3] Feature scope (STATIC — no YearGap):
  Numeric     : ['Expo']
  Categorical : ['Usage', 'VehType', 'VehPower']


C:\Users\User\AppData\Local\Temp\ipykernel_15784\1044886454.py:94: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train[col] = X_train[col].cat.set_categories(all_cats)
C:\Users\User\AppData\Local\Temp\ipykernel_15784\1044886454.py:95: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_val[col]   = X_val[col].cat.set_categories(all_cats)
C:\Users\User\AppData\Local\Temp\ipykernel_15784\1044886454.py:96: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[

[CHECK 3] X_train shape: (255241, 4)

[CHECK 4] scale_pos_weight: 6.90
[CHECK 4] Training LightGBM-Static...
Training until validation scores don't improve for 30 rounds
[50]	train's auc: 0.772031	val's auc: 0.765402
[100]	train's auc: 0.773979	val's auc: 0.765894
[150]	train's auc: 0.775155	val's auc: 0.766194
[200]	train's auc: 0.776091	val's auc: 0.766253
Early stopping, best iteration is:
[185]	train's auc: 0.775834	val's auc: 0.7663

[CHECK 4] Best iteration: 185

===== Test set evaluation (timestep-level) =====
Total timestep predictions: 54,755
AUC-ROC          : 0.7686
F1 @ thr=0.5      : 0.3603
Best threshold    : 0.636
Best F1           : 0.3777
  Precision       : 0.2902
  Recall          : 0.5408
Positive rate in test: 12.79%

===== Feature importance (gain) =====
 feature          gain
VehPower 735731.076576
 VehType 164542.415956
   Usage 128735.718308
    Expo  88042.382959

Saved: data/sequences/lightgbm_static_model.txt
Saved: data/sequences/lightgbm_static_test_predic